# Robust ML Pipeline: DivideMix + EM Label Shift + SAR TTA
### Fashion-MNIST | 30% Symmetric Label Noise | Covariate + Label Shift

**Pipeline Overview:**
- **Phase 1:** ResNet-18 trained from scratch using DivideMix with APL (Active Passive Loss = NCE + RCE) against 30% symmetric label noise
- **Phase 2:** EM-based Label Shift Estimation to compute class weight vector ŵ_t
- **Phase 3:** SAR Test-Time Adaptation using ŵ_t-weighted entropy objective

**Data files expected:**
- `source_toxic.pt` — noisy training data
- `val_sanity.pt` — small clean validation set
- `target_static.pt` — shifted + imbalanced target stream

## 0. Imports & Setup

In [ ]:
import os
import copy
import random
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader, Subset
import torchvision.transforms as transforms
from sklearn.mixture import GaussianMixture
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

# ── Reproducibility ──────────────────────────────────────────────────────────
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
NUM_CLASSES = 10
print(f'Using device: {DEVICE}')

## 1. Data Loading

In [ ]:
class TorchDataset(Dataset):
    """
    Generic wrapper for .pt files containing {'images'/'data': Tensor, 'labels'/'targets': Tensor}.
    Handles varying key names and missing labels (treats as unlabeled with dummy -1 labels).
    """
    def __init__(self, filepath, transform=None, mode='labeled', predict_transform=None):
        payload = torch.load(filepath, weights_only=False)

        # Handle varying key names for images
        if 'images' in payload:
            self.data = payload['images']
        elif 'data' in payload:
            self.data = payload['data']
        else:
            raise KeyError(f"No 'images' or 'data' key found. Available keys: {list(payload.keys())}")

        # Handle varying key names for labels (fallback to dummy -1)
        if 'labels' in payload:
            self.labels = payload['labels'].long()
        elif 'targets' in payload:
            self.labels = payload['targets'].long()
        elif 'label' in payload:
            self.labels = payload['label'].long()
        else:
            print(f"  ⚠ No label key found in {filepath}. Keys: {list(payload.keys())}. Using dummy labels (-1).")
            self.labels = torch.full((self.data.shape[0],), -1, dtype=torch.long)

        self.transform         = transform
        self.predict_transform = predict_transform if predict_transform else transform
        self.mode   = mode

        # Ensure data is (N,1,28,28) float in [0,1]
        if self.data.dim() == 3:
            self.data = self.data.unsqueeze(1)
        if self.data.dtype != torch.float32:
            self.data = self.data.float()
        if self.data.max() > 1.0:
            self.data = self.data / 255.0

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, idx):
        img = self.data[idx]   # (1,28,28)
        lbl = self.labels[idx]

        if self.mode == 'labeled':
            img_t = self.transform(img) if self.transform else img
            return img_t, lbl, idx

        elif self.mode == 'unlabeled':
            img1 = self.transform(img) if self.transform else img
            img2 = self.transform(img) if self.transform else img
            return img1, img2, idx

        elif self.mode == 'predict':
            img_t = self.predict_transform(img) if self.predict_transform else img
            return img_t, lbl, idx


# ── Transforms ───────────────────────────────────────────────────────────────
# Fashion-MNIST mean/std (grayscale)
FMNIST_MEAN = (0.2860,)
FMNIST_STD  = (0.3530,)

train_transform = transforms.Compose([
    transforms.RandomHorizontalFlip(),
    transforms.RandomCrop(28, padding=4),
    transforms.Normalize(FMNIST_MEAN, FMNIST_STD),
])

eval_transform = transforms.Compose([
    transforms.Normalize(FMNIST_MEAN, FMNIST_STD),
])

print('Transforms defined.')

In [ ]:
# ── Load datasets ─────────────────────────────────────────────────────────────
# Adjust paths as needed
import shutil

from google.colab import drive
drive.mount('/content/drive')

# ── Drive save directory for all artifacts ────────────────────────────────────
DRIVE_SAVE_DIR = '/content/drive/MyDrive/hackenza_results'
os.makedirs(DRIVE_SAVE_DIR, exist_ok=True)
print(f'All artifacts will be saved to: {DRIVE_SAVE_DIR}')

SOURCE_PATH  = '/content/drive/MyDrive/source_toxic.pt'
VAL_PATH     = '/content/drive/MyDrive/val_sanity.pt'
TARGET_PATH  = '/content/drive/MyDrive/static.pt'

source_dataset = TorchDataset(SOURCE_PATH,  transform=train_transform,   mode='labeled')
val_dataset    = TorchDataset(VAL_PATH,     transform=eval_transform,    mode='predict')
target_dataset = TorchDataset(TARGET_PATH,  transform=eval_transform,    mode='predict')

val_loader    = DataLoader(val_dataset,    batch_size=64,  shuffle=False, num_workers=2)
target_loader = DataLoader(target_dataset, batch_size=64,  shuffle=False, num_workers=2)

print(f'Source (toxic) samples : {len(source_dataset)}')
print(f'Val (sanity)   samples : {len(val_dataset)}')
print(f'Target (static) samples: {len(target_dataset)}')

## 2. Model: ResNet-18 (Modified for 28×28 Grayscale)

In [ ]:
class BasicBlock(nn.Module):
    expansion = 1

    def __init__(self, in_planes, planes, stride=1):
        super().__init__()
        self.conv1 = nn.Conv2d(in_planes, planes, 3, stride=stride, padding=1, bias=False)
        self.bn1   = nn.BatchNorm2d(planes)
        self.conv2 = nn.Conv2d(planes, planes, 3, stride=1, padding=1, bias=False)
        self.bn2   = nn.BatchNorm2d(planes)

        self.shortcut = nn.Sequential()
        if stride != 1 or in_planes != planes:
            self.shortcut = nn.Sequential(
                nn.Conv2d(in_planes, planes, 1, stride=stride, bias=False),
                nn.BatchNorm2d(planes)
            )

    def forward(self, x):
        out = F.relu(self.bn1(self.conv1(x)))
        out = self.bn2(self.conv2(out))
        out += self.shortcut(x)
        return F.relu(out)


class ResNet18_FMNIST(nn.Module):
    """
    ResNet-18 adapted for Fashion-MNIST (28×28 grayscale).
    Key changes vs standard ResNet-18:
      - Input conv: 3×3 kernel, stride 1 (not 7×7 stride 2)
      - No initial MaxPool (would destroy spatial info on 28×28)
      - Modified head: Linear -> BN -> ReLU -> Dropout(0.3) -> Linear
      - Kaiming uniform initialization throughout
    """
    def __init__(self, num_classes=10):
        super().__init__()
        # Stem: 3x3 conv, stride 1, no maxpool
        self.stem = nn.Sequential(
            nn.Conv2d(1, 64, kernel_size=3, stride=1, padding=1, bias=False),
            nn.BatchNorm2d(64),
            nn.ReLU(inplace=True)
        )
        # Residual layers
        self.layer1 = self._make_layer(64,  64,  2, stride=1)
        self.layer2 = self._make_layer(64,  128, 2, stride=2)
        self.layer3 = self._make_layer(128, 256, 2, stride=2)
        self.layer4 = self._make_layer(256, 512, 2, stride=2)
        self.avgpool = nn.AdaptiveAvgPool2d((1, 1))
        # Modified classification head
        self.head = nn.Sequential(
            nn.Linear(512, 256),
            nn.BatchNorm1d(256),
            nn.ReLU(inplace=True),
            nn.Dropout(0.3),
            nn.Linear(256, num_classes)
        )
        self._initialize_weights()

    def _make_layer(self, in_planes, planes, num_blocks, stride):
        strides = [stride] + [1] * (num_blocks - 1)
        layers  = []
        for s in strides:
            layers.append(BasicBlock(in_planes, planes, s))
            in_planes = planes
        return nn.Sequential(*layers)

    def _initialize_weights(self):
        for m in self.modules():
            if isinstance(m, nn.Conv2d):
                nn.init.kaiming_uniform_(m.weight, mode='fan_out', nonlinearity='relu')
            elif isinstance(m, nn.BatchNorm2d) or isinstance(m, nn.BatchNorm1d):
                nn.init.constant_(m.weight, 1)
                nn.init.constant_(m.bias, 0)
            elif isinstance(m, nn.Linear):
                nn.init.kaiming_uniform_(m.weight, nonlinearity='relu')
                if m.bias is not None:
                    nn.init.constant_(m.bias, 0)

    def forward(self, x):
        x = self.stem(x)
        x = self.layer1(x)
        x = self.layer2(x)
        x = self.layer3(x)
        x = self.layer4(x)
        x = self.avgpool(x)
        x = torch.flatten(x, 1)
        return self.head(x)


# Quick sanity check
dummy = torch.zeros(2, 1, 28, 28)
model_test = ResNet18_FMNIST()
out = model_test(dummy)
print(f'Output shape: {out.shape}  ✓  (expected: [2, 10])')
total_params = sum(p.numel() for p in model_test.parameters())
print(f'Total parameters: {total_params:,}')

## 3. Loss Functions

In [ ]:
class APLLoss(nn.Module):
    """
    Active Passive Loss (Ma et al., ICML 2020).
    Combines NCE (Normalized Cross-Entropy) + RCE (Reverse Cross-Entropy).

    APL = alpha * NCE + beta * RCE

    NCE(p, y) = CE(p, y) / Normalizer    — bounded active loss
    RCE(p, y) = -sum_k y_k * log(p_k)    — passive/reverse loss

    Both components are noise-tolerant:
    - NCE is bounded above (unlike CE), preventing unbounded memorization
    - RCE encourages the model to place probability on the correct class
      even under label noise

    Used as the supervised loss within DivideMix's clean-sample branch.
    """
    def __init__(self, alpha=1.0, beta=1.0, num_classes=10, reduction='mean'):
        super().__init__()
        self.alpha = alpha
        self.beta = beta
        self.num_classes = num_classes
        self.reduction = reduction

    def forward(self, logits, targets, weights=None):
        probs = F.softmax(logits, dim=1)
        probs = torch.clamp(probs, min=1e-7, max=1.0)  # numerical stability

        # Handle soft (one-hot-like) targets from MixUp
        if targets.dim() == 1:
            one_hot = F.one_hot(targets, self.num_classes).float()
        else:
            one_hot = targets

        # ── NCE: Normalized Cross-Entropy ────────────────────────────────
        # CE per sample = -sum_k y_k * log(p_k)
        ce = -(one_hot * torch.log(probs)).sum(dim=1)
        # Normalizer A = -sum_k log(p_k)
        normalizer = -torch.log(probs).sum(dim=1)
        nce = ce / (normalizer + 1e-7)

        # ── RCE: Reverse Cross-Entropy ───────────────────────────────────
        # RCE = -sum_k p_k * log(y_k + eps)
        # For one-hot y: only the y=1 entry contributes log(1)≈0,
        # other terms give -p_k * log(eps), penalizing probability on wrong classes
        rce = -(probs * torch.log(one_hot + 1e-4)).sum(dim=1)

        loss = self.alpha * nce + self.beta * rce

        if weights is not None:
            loss = loss * weights
        if self.reduction == 'mean':
            return loss.mean()
        return loss


def mixup_data(x, y_soft, alpha=0.5):
    """MixUp augmentation with soft labels."""
    lam = np.random.beta(alpha, alpha)
    idx = torch.randperm(x.size(0), device=x.device)
    mixed_x = lam * x + (1 - lam) * x[idx]
    mixed_y = lam * y_soft + (1 - lam) * y_soft[idx]
    return mixed_x, mixed_y


print('Loss functions defined (APL = NCE + RCE).')

## 4. Phase 1 — DivideMix Training

In [ ]:
# ── DivideMix Hyperparameters ─────────────────────────────────────────────────
WARMUP_EPOCHS   = 15       # was 5 — longer warmup gives stable GMM initialization
DIVIDEMIX_EPOCHS = 50      # was 25 — more DivideMix epochs (with early stopping)
TOTAL_EPOCHS    = WARMUP_EPOCHS + DIVIDEMIX_EPOCHS

BATCH_SIZE      = 128
LR_INIT         = 0.01     # was 0.02 — lower LR prevents overfitting to clean subset
LR_MIN          = 1e-4
MOMENTUM        = 0.9
WEIGHT_DECAY    = 5e-4     # was 1e-4 — stronger regularization

LAMBDA_U_MAX    = 5.0      # was 25.0 — much lower; 25 was destabilizing training
LAMBDA_U_RAMP   = 16       # epochs to ramp up lambda_u after warmup
TEMP_SHARP      = 0.5      # sharpening temperature for pseudo-labels
MIXUP_ALPHA     = 0.5
P_THRESHOLD     = 0.5      # BMM split threshold
APL_ALPHA       = 1.0      # weight for NCE component
APL_BETA        = 1.0      # weight for RCE component
MIN_CLEAN_RATIO = 0.4      # NEW: floor on clean sample ratio to prevent collapse
PATIENCE        = 10       # NEW: early stopping patience on val accuracy

print(f'Total training epochs: {TOTAL_EPOCHS} (with early stopping, patience={PATIENCE})')

In [ ]:
def sharpen(probs, T):
    """Sharpening function for pseudo-labels: lower T -> harder labels."""
    sharpened = probs.pow(1.0 / T)
    return sharpened / sharpened.sum(dim=1, keepdim=True)


def compute_per_sample_loss(model, dataset, device):
    """
    Compute per-sample CE loss for BMM fitting.
    Returns numpy array of shape (N,).
    """
    model.eval()
    loader = DataLoader(dataset, batch_size=256, shuffle=False, num_workers=2)
    losses = []
    with torch.no_grad():
        for imgs, labels, _ in loader:
            imgs, labels = imgs.to(device), labels.to(device)
            logits = model(imgs)
            loss = F.cross_entropy(logits, labels, reduction='none')
            losses.append(loss.cpu().numpy())
    return np.concatenate(losses)


def fit_bmm_and_split(losses, p_threshold=0.5):
    """
    Fit a two-component Gaussian Mixture on per-sample losses.
    Returns prob_clean: array of shape (N,) — probability each sample is clean.
    The clean component is identified as the one with the lower mean.

    FIX: Use standard GaussianMixture (BayesianGMM was collapsing components).
    FIX: Enforce MIN_CLEAN_RATIO floor to prevent feedback-loop collapse.
    """
    from sklearn.mixture import GaussianMixture

    losses_normed = (losses - losses.min()) / (losses.max() - losses.min() + 1e-8)
    losses_normed = losses_normed.reshape(-1, 1)

    gmm = GaussianMixture(
        n_components=2,
        max_iter=200,
        tol=1e-4,
        reg_covar=1e-6,
        random_state=SEED
    )
    gmm.fit(losses_normed)

    posteriors = gmm.predict_proba(losses_normed)  # (N, 2)
    # Clean component = lower mean
    means = gmm.means_.flatten()
    clean_idx = np.argmin(means)
    prob_clean = posteriors[:, clean_idx]

    # Enforce minimum clean ratio — prevents collapse
    n_clean = (prob_clean >= p_threshold).sum()
    min_clean = int(MIN_CLEAN_RATIO * len(losses))
    if n_clean < min_clean:
        # Lower the effective threshold until we have enough clean samples
        sorted_probs = np.sort(prob_clean)[::-1]
        adjusted_threshold = sorted_probs[min_clean - 1]
        print(f'  [GMM] Clean count {n_clean} < floor {min_clean}, '
              f'adjusting threshold from {p_threshold:.3f} to {adjusted_threshold:.3f}')
        # Boost prob_clean for borderline samples to maintain the floor
        prob_clean = np.maximum(prob_clean, (prob_clean >= adjusted_threshold).astype(float) * p_threshold)

    return prob_clean


print('DivideMix utilities defined (with GaussianMixture + clean ratio floor).')

In [ ]:
def warmup_epoch(model, optimizer, loader, device, apl_loss_fn):
    """
    Single warm-up epoch.
    FIX: Use APL loss instead of raw CE — CE memorizes noisy labels during warmup,
    poisoning the loss distribution before GMM even starts.
    FIX: Add gradient clipping for stability.
    """
    model.train()
    total_loss, correct, total = 0.0, 0, 0
    for imgs, labels, _ in loader:
        imgs, labels = imgs.to(device), labels.to(device)
        optimizer.zero_grad()
        logits = model(imgs)
        loss = apl_loss_fn(logits, labels)  # APL (NCE + RCE)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        optimizer.step()
        total_loss += loss.item() * imgs.size(0)
        correct    += (logits.argmax(1) == labels).sum().item()
        total      += imgs.size(0)
    train_acc = correct / total
    return total_loss / total, train_acc


def dividemix_epoch(net1, net2, optimizer1, optimizer2,
                    dataset, prob_clean1, prob_clean2,
                    lambda_u, device, apl_loss_fn, epoch):
    net1.train()
    net2.eval()

    clean_mask    = prob_clean2 >= P_THRESHOLD
    noisy_mask    = ~clean_mask
    clean_indices = np.where(clean_mask)[0]
    noisy_indices = np.where(noisy_mask)[0]

    if len(clean_indices) == 0:
        clean_indices = np.arange(len(dataset))

    clean_set = TorchDataset.__new__(TorchDataset)
    clean_set.data              = dataset.data[clean_indices]
    clean_set.labels            = dataset.labels[clean_indices]
    clean_set.transform         = dataset.transform
    clean_set.predict_transform = dataset.transform
    clean_set.mode              = 'labeled'
    clean_loader = DataLoader(clean_set, batch_size=BATCH_SIZE,
                              shuffle=True, num_workers=2, drop_last=True)

    total_loss, total_sup, total_unsup = 0.0, 0.0, 0.0
    correct, total_samples, n_batches  = 0, 0, 0

    if len(noisy_indices) > BATCH_SIZE and lambda_u > 0:
        noisy_set = TorchDataset.__new__(TorchDataset)
        noisy_set.data              = dataset.data[noisy_indices]
        noisy_set.labels            = dataset.labels[noisy_indices]
        noisy_set.transform         = dataset.transform
        noisy_set.predict_transform = dataset.transform
        noisy_set.mode              = 'unlabeled'
        noisy_loader = DataLoader(noisy_set, batch_size=BATCH_SIZE,
                                  shuffle=True, num_workers=2, drop_last=True)
        noisy_iter = iter(noisy_loader)
    else:
        noisy_iter = None

    for imgs_c, labels_c, _ in clean_loader:
        imgs_c, labels_c = imgs_c.to(device), labels_c.to(device)

        labels_soft          = F.one_hot(labels_c, NUM_CLASSES).float()
        imgs_mix, labels_mix = mixup_data(imgs_c, labels_soft, MIXUP_ALPHA)

        optimizer1.zero_grad()
        logits_mix = net1(imgs_mix)
        loss_sup   = apl_loss_fn(logits_mix, labels_mix)

        loss_unsup = torch.tensor(0.0, device=device)
        if noisy_iter is not None:
            try:
                imgs_u1, imgs_u2, _ = next(noisy_iter)
            except StopIteration:
                noisy_iter = iter(noisy_loader)
                imgs_u1, imgs_u2, _ = next(noisy_iter)

            imgs_u1, imgs_u2 = imgs_u1.to(device), imgs_u2.to(device)
            with torch.no_grad():
                p1     = torch.softmax(net2(imgs_u1), dim=1)
                p2     = torch.softmax(net2(imgs_u2), dim=1)
                pseudo = sharpen((p1 + p2) / 2, TEMP_SHARP)

            logits_u1  = net1(imgs_u1)
            logits_u2  = net1(imgs_u2)
            loss_unsup = (
                F.kl_div(F.log_softmax(logits_u1, dim=1), pseudo, reduction='batchmean') +
                F.kl_div(F.log_softmax(logits_u2, dim=1), pseudo, reduction='batchmean')
            ) / 2

        loss = loss_sup + lambda_u * loss_unsup
        loss.backward()
        # FIX: gradient clipping prevents unstable parameter updates
        torch.nn.utils.clip_grad_norm_(net1.parameters(), max_norm=1.0)
        optimizer1.step()

        # Training accuracy on clean samples (against original labels, not mixed)
        with torch.no_grad():
            preds         = net1(imgs_c).argmax(dim=1)
            correct      += (preds == labels_c).sum().item()
            total_samples += labels_c.size(0)

        total_loss  += loss.item()
        total_sup   += loss_sup.item()
        total_unsup += loss_unsup.item()
        n_batches   += 1

    train_acc = correct / max(total_samples, 1)

    return (total_loss  / max(n_batches, 1),
            total_sup   / max(n_batches, 1),
            total_unsup / max(n_batches, 1),
            len(clean_indices),
            train_acc)

print('Training epoch functions defined (with APL warmup + gradient clipping).')

In [ ]:
@torch.no_grad()
def evaluate(model, loader, device):
    """Evaluate model accuracy on a labeled loader."""
    model.eval()
    correct, total = 0, 0
    for imgs, labels, _ in loader:
        imgs, labels = imgs.to(device), labels.to(device)
        preds = model(imgs).argmax(dim=1)
        correct += (preds == labels).sum().item()
        total   += labels.size(0)
    return correct / total if total > 0 else 0.0


print('Evaluate function defined.')

In [ ]:
# ── Initialize two networks for DivideMix ────────────────────────────────────
net1 = ResNet18_FMNIST(NUM_CLASSES).to(DEVICE)
net2 = ResNet18_FMNIST(NUM_CLASSES).to(DEVICE)

opt1 = optim.SGD(net1.parameters(), lr=LR_INIT, momentum=MOMENTUM, weight_decay=WEIGHT_DECAY)
opt2 = optim.SGD(net2.parameters(), lr=LR_INIT, momentum=MOMENTUM, weight_decay=WEIGHT_DECAY)

# Cosine annealing over total epochs
sched1 = optim.lr_scheduler.CosineAnnealingLR(opt1, T_max=TOTAL_EPOCHS, eta_min=LR_MIN)
sched2 = optim.lr_scheduler.CosineAnnealingLR(opt2, T_max=TOTAL_EPOCHS, eta_min=LR_MIN)

apl_fn = APLLoss(alpha=APL_ALPHA, beta=APL_BETA, num_classes=NUM_CLASSES).to(DEVICE)

# Warm-up loader (labeled mode)
warmup_loader = DataLoader(source_dataset, batch_size=BATCH_SIZE,
                           shuffle=True, num_workers=2, drop_last=True)

# History
history = {'epoch': [], 'val_acc1': [], 'val_acc2': [],
           'loss': [], 'n_clean': [], 'lambda_u': []}

print(f'Models initialized. LR={LR_INIT}, WD={WEIGHT_DECAY}, λ_u_max={LAMBDA_U_MAX}')

In [ ]:
# ── Training Loop (with early stopping) ───────────────────────────────────────
prob_clean1 = np.ones(len(source_dataset))  # initial: treat all as clean
prob_clean2 = np.ones(len(source_dataset))

best_val_acc = 0.0
best_state1  = None
patience_counter = 0  # NEW: early stopping counter

for epoch in range(1, TOTAL_EPOCHS + 1):

    # ── WARM-UP PHASE ────────────────────────────────────────────────────────
    if epoch <= WARMUP_EPOCHS:
        loss1, train_acc1 = warmup_epoch(net1, opt1, warmup_loader, DEVICE, apl_fn)
        loss2, train_acc2 = warmup_epoch(net2, opt2, warmup_loader, DEVICE, apl_fn)
        loss_total  = (loss1 + loss2) / 2
        train_acc   = (train_acc1 + train_acc2) / 2
        n_clean     = len(source_dataset)
        lu          = 0.0

    # ── DIVIDEMIX PHASE ──────────────────────────────────────────────────────
    else:
        dm_epoch = epoch - WARMUP_EPOCHS
        lu = LAMBDA_U_MAX * min(1.0, dm_epoch / LAMBDA_U_RAMP)

        losses1     = compute_per_sample_loss(net1, source_dataset, DEVICE)
        losses2     = compute_per_sample_loss(net2, source_dataset, DEVICE)
        prob_clean1 = fit_bmm_and_split(losses1, P_THRESHOLD)
        prob_clean2 = fit_bmm_and_split(losses2, P_THRESHOLD)

        l1, ls1, lu1, nc1, ta1 = dividemix_epoch(
            net1, net2, opt1, opt2,
            source_dataset, prob_clean1, prob_clean2,
            lu, DEVICE, apl_fn, epoch
        )
        l2, ls2, lu2, nc2, ta2 = dividemix_epoch(
            net2, net1, opt2, opt1,
            source_dataset, prob_clean2, prob_clean1,
            lu, DEVICE, apl_fn, epoch
        )
        loss_total = (l1 + l2) / 2
        train_acc  = (ta1 + ta2) / 2
        n_clean    = (nc1 + nc2) // 2

    sched1.step()
    sched2.step()

    # ── Validation ───────────────────────────────────────────────────────────
    val_acc1 = evaluate(net1, val_loader, DEVICE)
    val_acc2 = evaluate(net2, val_loader, DEVICE)
    best_val = max(val_acc1, val_acc2)

    if best_val > best_val_acc:
        best_val_acc = best_val
        best_state1  = copy.deepcopy(net1.state_dict() if val_acc1 >= val_acc2
                                     else net2.state_dict())
        patience_counter = 0  # reset on improvement
    else:
        patience_counter += 1

    history['epoch'].append(epoch)
    history['val_acc1'].append(val_acc1)
    history['val_acc2'].append(val_acc2)
    history['loss'].append(loss_total)
    history['n_clean'].append(n_clean)
    history['lambda_u'].append(lu)

    if epoch % 5 == 0 or epoch <= WARMUP_EPOCHS:
        phase = 'WARMUP' if epoch <= WARMUP_EPOCHS else 'DIVIDEMIX'
        print(f'[{phase}] Epoch {epoch:3d}/{TOTAL_EPOCHS} | '
              f'Loss: {loss_total:.4f} | '
              f'Train Acc: {train_acc:.3f} | '
              f'Val Acc: {val_acc1:.3f}/{val_acc2:.3f} | '
              f'Clean: {n_clean}/{len(source_dataset)} | '
              f'λ_u: {lu:.1f} | '
              f'Patience: {patience_counter}/{PATIENCE}')

    # ── Early stopping ───────────────────────────────────────────────────────
    if patience_counter >= PATIENCE and epoch > WARMUP_EPOCHS:
        print(f'\n⚡ Early stopping at epoch {epoch} (no val improvement for {PATIENCE} epochs)')
        break

print(f'\nTraining complete. Best val acc: {best_val_acc:.4f}')

# Load best model
net1.load_state_dict(best_state1)
torch.save(best_state1, 'best_model_phase1.pt')
shutil.copy('best_model_phase1.pt', os.path.join(DRIVE_SAVE_DIR, 'best_model_phase1.pt'))
print('Best model saved to best_model_phase1.pt + Drive')

# Save training history
torch.save(history, 'training_history.pt')
shutil.copy('training_history.pt', os.path.join(DRIVE_SAVE_DIR, 'training_history.pt'))
print('Training history saved to Drive')

In [ ]:
# ── Training curves ───────────────────────────────────────────────────────────
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

axes[0].plot(history['epoch'], history['val_acc1'], label='Net1', alpha=0.8)
axes[0].plot(history['epoch'], history['val_acc2'], label='Net2', alpha=0.8)
axes[0].axvline(WARMUP_EPOCHS, color='red', linestyle='--', alpha=0.5, label='DivideMix start')
axes[0].set_title('Validation Accuracy')
axes[0].set_xlabel('Epoch')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

axes[1].plot(history['epoch'], history['loss'], color='orange')
axes[1].axvline(WARMUP_EPOCHS, color='red', linestyle='--', alpha=0.5)
axes[1].set_title('Training Loss')
axes[1].set_xlabel('Epoch')
axes[1].grid(True, alpha=0.3)

axes[2].plot(history['epoch'][WARMUP_EPOCHS:],
             history['n_clean'][WARMUP_EPOCHS:], color='green')
axes[2].axhline(len(source_dataset) * 0.7, color='red', linestyle='--',
                alpha=0.5, label='Expected clean (70%)')
axes[2].set_title('Clean Samples Identified by BMM')
axes[2].set_xlabel('Epoch')
axes[2].legend()
axes[2].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('phase1_training_curves.png', dpi=120, bbox_inches='tight')
shutil.copy('phase1_training_curves.png', os.path.join(DRIVE_SAVE_DIR, 'phase1_training_curves.png'))
plt.show()
print('Training curves saved to Drive')

## 5. Post-Training Calibration (Temperature Scaling)

In [ ]:
class TemperatureScaler(nn.Module):
    """
    Post-hoc calibration via a single temperature scalar T.
    Fitted by minimizing NLL on the clean val_sanity.pt set using LBFGS.
    Required before Phase 2 MLLS for reliable probability estimates.
    """
    def __init__(self, model):
        super().__init__()
        self.model = model
        self.temperature = nn.Parameter(torch.ones(1) * 1.5)

    def forward(self, x):
        return self.model(x) / self.temperature

    @torch.no_grad()
    def get_calibrated_probs(self, x):
        return F.softmax(self.forward(x), dim=1)


def fit_temperature(model, val_loader, device, max_iter=100):
    """
    Fit temperature T on the clean validation set.
    Freezes all model weights — only T is optimized.
    """
    model.eval()
    calibrated = TemperatureScaler(model).to(device)

    # Collect all logits and labels first
    all_logits, all_labels = [], []
    with torch.no_grad():
        for imgs, labels, _ in val_loader:
            imgs = imgs.to(device)
            logits = model(imgs)
            all_logits.append(logits)
            all_labels.append(labels.to(device))
    all_logits = torch.cat(all_logits)
    all_labels = torch.cat(all_labels)

    nll_before = F.cross_entropy(all_logits, all_labels).item()

    # Optimize only temperature
    optimizer = optim.LBFGS([calibrated.temperature], lr=0.01, max_iter=max_iter)

    def eval_fn():
        optimizer.zero_grad()
        loss = F.cross_entropy(all_logits / calibrated.temperature, all_labels)
        loss.backward()
        return loss

    optimizer.step(eval_fn)

    T = calibrated.temperature.item()
    nll_after = F.cross_entropy(all_logits / calibrated.temperature.detach(),
                                all_labels).item()

    print(f'Temperature T = {T:.4f}')
    print(f'NLL before calibration: {nll_before:.4f}')
    print(f'NLL after  calibration: {nll_after:.4f}')
    return calibrated


calibrated_model = fit_temperature(net1, val_loader, DEVICE)
torch.save({'model_state': net1.state_dict(),
            'temperature': calibrated_model.temperature.item()},
           'calibrated_model.pt')
shutil.copy('calibrated_model.pt', os.path.join(DRIVE_SAVE_DIR, 'calibrated_model.pt'))
print('Calibrated model saved to Drive')

## 6. Phase 2 — EM Label Shift Estimation

In [ ]:
def collect_softmax_probs(model, loader, device):
    """
    Collect calibrated softmax probabilities over all samples in loader.
    Returns: probs (N, C), labels (N,)
    """
    model.eval()
    all_probs, all_labels = [], []
    with torch.no_grad():
        for imgs, labels, _ in loader:
            imgs = imgs.to(device)
            probs = calibrated_model.get_calibrated_probs(imgs)
            all_probs.append(probs.cpu())
            all_labels.append(labels)
    return torch.cat(all_probs).numpy(), torch.cat(all_labels).numpy()


def em_label_shift(
    target_probs,          # (N_t, C) calibrated probs on target
    source_label_freq,     # (C,) empirical label frequencies from training
    num_classes=10,
    max_iter=50,
    tol=1e-4,
    verbose=True
):
    """
    EM-based Label Shift Estimation (Alexandari et al., 2020 / Lipton et al., 2018).

    Algorithm:
      Initialize q(y) = uniform
      E-step: w(y) = q(y) / p_s(y)
              compute per-sample posteriors using w(y) to reweight class priors
      M-step: q(y) = mean of per-sample posteriors over target set
      Repeat until L1 change in q(y) < tol

    Returns:
      q_target: (C,) estimated target label distribution
      weights:  (C,) importance weights w(y) = q_target(y) / p_source(y)
    """
    C = num_classes
    p_source = np.array(source_label_freq, dtype=np.float64)
    p_source = p_source / p_source.sum()  # normalize

    q = np.ones(C, dtype=np.float64) / C  # uniform initialization

    for iteration in range(max_iter):
        q_old = q.copy()

        # E-step: reweight class priors, compute posteriors
        # p(y|x, q) ∝ p(x|y) * q(y)  — implemented via:
        # posterior(y|x) = p_cal(y|x) * (q(y)/p_s(y)) / Z
        w = q / (p_source + 1e-10)
        weighted_probs = target_probs * w[np.newaxis, :]  # (N, C)
        weighted_probs = weighted_probs / (weighted_probs.sum(axis=1, keepdims=True) + 1e-10)

        # M-step: update q as mean posterior
        q = weighted_probs.mean(axis=0)
        q = q / q.sum()  # normalize

        delta = np.abs(q - q_old).sum()
        if verbose and (iteration % 10 == 0 or delta < tol):
            print(f'  EM iter {iteration+1:3d}: L1 change = {delta:.6f}')

        if delta < tol:
            print(f'  Converged at iteration {iteration+1}')
            break

    weights = q / (p_source + 1e-10)
    weights = weights / weights.mean()  # normalize so mean weight = 1

    return q, weights


print('EM label shift functions defined.')

In [ ]:
# ── Run Phase 2 ───────────────────────────────────────────────────────────────
print('Collecting calibrated probabilities on target stream...')
target_probs, target_labels_eval = collect_softmax_probs(
    calibrated_model, target_loader, DEVICE
)
print(f'Target probs shape: {target_probs.shape}')

# Source label frequencies (balanced: 0.1 each for Fashion-MNIST)
source_freq = np.ones(NUM_CLASSES) / NUM_CLASSES

print('\nRunning EM label shift estimation...')
q_target, w_hat = em_label_shift(
    target_probs,
    source_freq,
    num_classes=NUM_CLASSES,
    max_iter=50,
    tol=1e-4,
    verbose=True
)

print(f'\nEstimated target label distribution q(y):')
class_names = ['T-shirt', 'Trouser', 'Pullover', 'Dress', 'Coat',
               'Sandal', 'Shirt', 'Sneaker', 'Bag', 'Ankle boot']
for i, (c, q, w) in enumerate(zip(class_names, q_target, w_hat)):
    print(f'  {c:12s}: q={q:.4f}  w={w:.4f}')

# Save weights for Phase 3
w_hat_tensor = torch.tensor(w_hat, dtype=torch.float32)
torch.save(w_hat_tensor, 'label_shift_weights.pt')
shutil.copy('label_shift_weights.pt', os.path.join(DRIVE_SAVE_DIR, 'label_shift_weights.pt'))

# Save target probabilities and label shift results
torch.save({
    'target_probs': torch.tensor(target_probs),
    'q_target': q_target,
    'w_hat': w_hat,
    'source_freq': source_freq,
    'class_names': class_names,
}, 'label_shift_results.pt')
shutil.copy('label_shift_results.pt', os.path.join(DRIVE_SAVE_DIR, 'label_shift_results.pt'))
print('\nLabel shift weights + results saved to Drive')

In [ ]:
# ── Visualize label shift ─────────────────────────────────────────────────────
x = np.arange(NUM_CLASSES)
width = 0.35
fig, ax = plt.subplots(figsize=(12, 4))
ax.bar(x - width/2, source_freq, width, label='Source p_s(y)', alpha=0.8, color='steelblue')
ax.bar(x + width/2, q_target,   width, label='Target q(y) [EM est.]', alpha=0.8, color='tomato')
ax.set_xticks(x)
ax.set_xticklabels(class_names, rotation=45, ha='right')
ax.set_title('Label Shift: Source vs Estimated Target Distribution')
ax.legend()
ax.grid(True, alpha=0.3, axis='y')
plt.tight_layout()
plt.savefig('phase2_label_shift.png', dpi=120, bbox_inches='tight')
shutil.copy('phase2_label_shift.png', os.path.join(DRIVE_SAVE_DIR, 'phase2_label_shift.png'))
plt.show()
print('Label shift plot saved to Drive')

## 7. Phase 3 — SAR Test-Time Adaptation with ŵ_t-Weighted Entropy

In [ ]:
# ── Recreate target loader with num_workers=0 for Colab stability ─────────────
target_loader = DataLoader(target_dataset, batch_size=64, shuffle=False, num_workers=0)

# ── Weighted entropy utility ──────────────────────────────────────────────────
def weighted_entropy(probs, weights):
    log_probs = torch.log(probs + 1e-8)
    weighted  = weights.unsqueeze(0) * probs * (-log_probs)
    return weighted.sum(dim=1)


# ── SARAdapter class ──────────────────────────────────────────────────────────
class SARAdapter:
    """
    FIX: entropy_threshold raised from 0.4*ln(K) to 0.7*ln(K).
    The old 0.4 gate (0.921) blocked nearly ALL samples since batch entropies
    were ~1.4-1.6. With 0.7*ln(10)=1.612, most samples pass the filter,
    allowing actual BN adaptation to happen.
    FIX: LR lowered from 1e-3 to 5e-4 for more stable BN updates.
    """
    def __init__(self, model, weights, lr=5e-4, entropy_threshold=None,
                 num_classes=10, device='cpu'):
        self.model   = copy.deepcopy(model)
        self.model.to(device)
        self.device  = device
        self.weights = weights.to(device)

        if entropy_threshold is None:
            self.entropy_threshold = 0.7 * np.log(num_classes)  # was 0.4
        else:
            self.entropy_threshold = entropy_threshold
        print(f'SAR entropy threshold: {self.entropy_threshold:.4f}')

        self.bn_params = []
        for module in self.model.modules():
            if isinstance(module, (nn.BatchNorm2d, nn.BatchNorm1d)):
                module.requires_grad_(True)
                module.track_running_stats = True
                module.num_batches_tracked.zero_()
                self.bn_params.extend([module.weight, module.bias])

        for name, param in self.model.named_parameters():
            if not any(param is p for p in self.bn_params):
                param.requires_grad_(False)

        self.optimizer = optim.SGD(self.bn_params, lr=lr, momentum=0.9)
        print(f'SAR: adapting {len(self.bn_params)} BN parameters, lr={lr}')

    @torch.no_grad()
    def predict(self, x):
        self.model.eval()
        return F.softmax(self.model(x), dim=1)

    def adapt_and_predict(self, x):
        self.model.train()
        for module in self.model.modules():
            if isinstance(module, nn.BatchNorm1d):
                module.eval()

        self.optimizer.zero_grad()

        with torch.no_grad():
            probs_no_grad = F.softmax(self.model(x), dim=1)
            H_std = -(probs_no_grad * torch.log(probs_no_grad + 1e-8)).sum(dim=1)
            reliable_mask = H_std < self.entropy_threshold

        if reliable_mask.sum() <= 1:
            self.model.eval()
            with torch.no_grad():
                return F.softmax(self.model(x), dim=1)

        x_reliable  = x[reliable_mask]
        probs_adapt = F.softmax(self.model(x_reliable), dim=1)
        H_weighted  = weighted_entropy(probs_adapt, self.weights)
        loss        = H_weighted.mean()
        loss.backward()
        self.optimizer.step()

        self.model.eval()
        with torch.no_grad():
            final_probs = F.softmax(self.model(x), dim=1)
        return final_probs


# ── Initialize SAR ────────────────────────────────────────────────────────────
w_hat_tensor = torch.load('label_shift_weights.pt')
print(f'Loaded ŵ_t: {w_hat_tensor.numpy().round(3)}')

sar = SARAdapter(
    model=net1,
    weights=w_hat_tensor,
    lr=5e-4,           # was 1e-3
    num_classes=NUM_CLASSES,
    device=DEVICE
)

# ── Baseline (no TTA) ─────────────────────────────────────────────────────────
net1.eval()
correct_baseline, total = 0, 0
with torch.no_grad():
    for imgs, labels, _ in target_loader:
        imgs, labels = imgs.to(DEVICE), labels.to(DEVICE)
        preds = net1(imgs).argmax(dim=1)
        correct_baseline += (preds == labels).sum().item()
        total += labels.size(0)
acc_baseline = correct_baseline / total
print(f'Baseline accuracy (no TTA): {acc_baseline:.4f}')

# ── SAR TTA evaluation ────────────────────────────────────────────────────────
correct_sar, total_sar = 0, 0
entropy_log = []

for imgs, labels, _ in target_loader:
    imgs, labels = imgs.to(DEVICE), labels.to(DEVICE)
    probs = sar.adapt_and_predict(imgs)
    preds = probs.argmax(dim=1)
    correct_sar += (preds == labels).sum().item()
    total_sar   += labels.size(0)
    H = -(probs * torch.log(probs + 1e-8)).sum(dim=1).mean().item()
    entropy_log.append(H)

acc_sar = correct_sar / total_sar
print(f'SAR TTA accuracy (with ŵ_t): {acc_sar:.4f}')
print(f'Improvement: +{(acc_sar - acc_baseline)*100:.2f}%')

# ── Save SAR adapted model to Drive ──────────────────────────────────────────
torch.save(sar.model.state_dict(), 'sar_adapted_model.pt')
shutil.copy('sar_adapted_model.pt', os.path.join(DRIVE_SAVE_DIR, 'sar_adapted_model.pt'))
print('SAR adapted model saved to Drive')

# ── Entropy over adaptation stream ───────────────────────────────────────────
ENT_GATE_COEFF = 0.7  # matches the new threshold
plt.figure(figsize=(10, 4))
plt.plot(entropy_log, alpha=0.7, color='purple')
plt.axhline(ENT_GATE_COEFF * np.log(NUM_CLASSES), color='red', linestyle='--',
            label=f'Entropy gate ({ENT_GATE_COEFF*np.log(NUM_CLASSES):.3f})')
plt.xlabel('Batch index')
plt.ylabel('Mean batch entropy')
plt.title('SAR: Entropy over Target Stream (lower = more confident adaptation)')
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig('phase3_entropy.png', dpi=120, bbox_inches='tight')
shutil.copy('phase3_entropy.png', os.path.join(DRIVE_SAVE_DIR, 'phase3_entropy.png'))
plt.show()
print('Entropy plot saved to Drive')

## 8. Full Results Summary

In [ ]:
print('=' * 55)
print('           PIPELINE RESULTS SUMMARY')
print('=' * 55)
print(f'  Architecture      : ResNet-18 (modified, from scratch)')
print(f'  Phase 1 method    : DivideMix + APL (NCE α={APL_ALPHA}, RCE β={APL_BETA})')
print(f'  Training epochs   : {TOTAL_EPOCHS} ({WARMUP_EPOCHS} warmup + {DIVIDEMIX_EPOCHS} DivideMix)')
print(f'  Best val accuracy : {best_val_acc:.4f}')
print('-' * 55)
print(f'  Phase 2 method    : EM Label Shift Estimation')
print(f'  Max label weight  : {w_hat.max():.4f} (class {w_hat.argmax()})')
print(f'  Min label weight  : {w_hat.min():.4f} (class {w_hat.argmin()})')
print('-' * 55)
print(f'  Phase 3 method    : SAR + ŵ_t weighted entropy')
print(f'  Baseline acc (target, no TTA): {acc_baseline:.4f}')
print(f'  SAR TTA acc (target)         : {acc_sar:.4f}')
print(f'  TTA improvement              : +{(acc_sar - acc_baseline)*100:.2f}%')
print('=' * 55)

## 9. Save Final Model for Submission

In [ ]:
torch.save({
    'model_state_dict'  : sar.model.state_dict(),
    'temperature'       : calibrated_model.temperature.item(),
    'label_shift_weights': w_hat_tensor,
    'q_target'          : q_target,
    'source_label_freq' : source_freq,
    'config': {
        'num_classes'   : NUM_CLASSES,
        'apl_alpha'     : APL_ALPHA,
        'apl_beta'      : APL_BETA,
        'warmup_epochs' : WARMUP_EPOCHS,
        'total_epochs'  : TOTAL_EPOCHS,
        'sar_lr'        : 5e-4,
        'entropy_thresh': float(0.7 * np.log(NUM_CLASSES)),
    },
    'results': {
        'best_val_acc'  : best_val_acc,
        'baseline_target_acc': acc_baseline,
        'sar_target_acc': acc_sar,
    }
}, 'final_pipeline.pt')
shutil.copy('final_pipeline.pt', os.path.join(DRIVE_SAVE_DIR, 'final_pipeline.pt'))

print('Final pipeline saved to final_pipeline.pt + Drive')
print('Artifacts: phase1_training_curves.png, phase2_label_shift.png, phase3_entropy.png')

In [ ]:
from google.colab import files

# Download the best Phase 1 model
files.download('best_model_phase1.pt')

# Download the full final pipeline (model + weights + config)
files.download('final_pipeline.pt')

# Download the label shift weights separately
files.download('label_shift_weights.pt')

# Download the training curves
files.download('phase1_training_curves.png')
files.download('phase2_label_shift.png')
files.download('phase3_entropy.png')

## 10. Test Suite Evaluation (test_suite_public.pt)

In [ ]:

import csv

# ══════════════════════════════════════════════════════════════════════════════
# PART A: Predict on target_static.pt  (IDs: static_0, static_1, …)
# ══════════════════════════════════════════════════════════════════════════════
print('Predicting on target_static (for static_i rows)...')
sar_static = SARAdapter(
    model=net1, weights=w_hat_tensor, lr=5e-4,
    num_classes=NUM_CLASSES, device=DEVICE
)

static_preds = []
static_loader_sub = DataLoader(target_dataset, batch_size=64, shuffle=False, num_workers=0)
for imgs, _, _ in static_loader_sub:
    imgs = imgs.to(DEVICE)
    probs = sar_static.adapt_and_predict(imgs)
    static_preds.append(probs.argmax(dim=1).cpu())
static_preds = torch.cat(static_preds)
print(f'  Static predictions: {len(static_preds)} samples')

# Save static predictions to Drive
torch.save(static_preds, 'static_preds.pt')
shutil.copy('static_preds.pt', os.path.join(DRIVE_SAVE_DIR, 'static_preds.pt'))
print('  Static predictions saved to Drive')

# ══════════════════════════════════════════════════════════════════════════════
# PART B: Predict on test_suite_public.pt  (IDs: scenario_XX_0, scenario_XX_1, …)
# ══════════════════════════════════════════════════════════════════════════════
TEST_SUITE_PATH = '/content/drive/MyDrive/test_suite_public.pt'
test_suite = torch.load(TEST_SUITE_PATH, weights_only=False)
print(f'\nTest suite loaded: {len(test_suite)} scenarios')
print(f'Scenario keys: {sorted(test_suite.keys())}\n')

all_predictions = {}

for scenario_name in sorted(test_suite.keys()):
    scenario_data = test_suite[scenario_name]

    if isinstance(scenario_data, dict):
        images = scenario_data['images']
    else:
        images = scenario_data

    if images.dim() == 3:
        images = images.unsqueeze(1)
    if images.dtype != torch.float32:
        images = images.float()
    if images.max() > 1.0:
        images = images / 255.0

    images_normed = (images - FMNIST_MEAN[0]) / FMNIST_STD[0]

    sar_scenario = SARAdapter(
        model=net1, weights=w_hat_tensor, lr=5e-4,
        num_classes=NUM_CLASSES, device=DEVICE
    )

    scenario_preds = []
    for start in range(0, len(images_normed), 64):
        batch = images_normed[start:start + 64].to(DEVICE)
        probs = sar_scenario.adapt_and_predict(batch)
        scenario_preds.append(probs.argmax(dim=1).cpu())

    scenario_preds = torch.cat(scenario_preds)
    all_predictions[scenario_name] = scenario_preds

    class_counts = torch.bincount(scenario_preds, minlength=NUM_CLASSES)
    print(f'  {scenario_name}: {len(images)} imgs, top class: {class_names[class_counts.argmax().item()]}')

# Save all scenario predictions to Drive
torch.save(all_predictions, 'all_scenario_predictions.pt')
shutil.copy('all_scenario_predictions.pt', os.path.join(DRIVE_SAVE_DIR, 'all_scenario_predictions.pt'))
print('\nAll scenario predictions saved to Drive')

# ══════════════════════════════════════════════════════════════════════════════
# PART C: Build submission.csv  —  columns: ID, Category
# ══════════════════════════════════════════════════════════════════════════════
csv_path = 'submission.csv'
row_count = 0

with open(csv_path, 'w', newline='') as f:
    writer = csv.writer(f)
    writer.writerow(['ID', 'Category'])

    # static_i rows
    for i, pred in enumerate(static_preds):
        writer.writerow([f'static_{i}', pred.item()])
        row_count += 1

    # scenario_XX_i rows
    for scenario_name in sorted(all_predictions.keys()):
        preds = all_predictions[scenario_name]
        for i, pred in enumerate(preds):
            writer.writerow([f'{scenario_name}_{i}', pred.item()])
            row_count += 1

print(f'\n✅ submission.csv saved — {row_count} rows')
print(f'   static rows   : {len(static_preds)}')
print(f'   scenario rows  : {sum(len(p) for p in all_predictions.values())}')

# Save submission.csv to Drive
shutil.copy(csv_path, os.path.join(DRIVE_SAVE_DIR, 'submission.csv'))
print(f'   submission.csv copied to Drive')

# ── Download ──────────────────────────────────────────────────────────────────
from google.colab import files
files.download(csv_path)